# 01 — Base model outputs (Step 1)
Produces, from a **fresh kernel, top to bottom**:
- `split_v1.json` — frozen patient-level split + 5 GroupKFold folds
- `images_uint8_224.pt` — cached raw images (no augmentation baked in)
- `xgb_final_v1.joblib`, `vit_final_v1.pt` — the two branch models we will explain
- `base_outputs_v1.npz` — OOF + test probabilities for both branches

ViT recipe follows Imrie et al. (IEEE JBHI 2025, Appendix A2): stage 1 head-only lr 1e-4 (≤50 ep), stage 2 full backbone lr 1e-6, weighted CE, early stopping on inner-val loss (patience 20). **The same function is used in every fold and for the final model.**

Before running: delete `oof_predictions.npz` and the old `cache/` folder.

In [9]:
import json, copy, joblib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from PIL import Image
from tqdm import tqdm
from torchvision import models
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

from common import META, IMG_ROOT, FEATURES, TARGET, build_image_index

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('device:', device)

device: cuda


## 1a. Frozen split (identical logic to edaV2, but saved to disk)

In [10]:
data = pd.read_csv(META)
data[TARGET] = data[TARGET].replace({'BOD': 'SCC'})
data = data.sort_values('patient_id').reset_index(drop=True)

patients = np.asarray(data['patient_id'].unique())          # numpy array, same order edaV2 used
tr_p, te_p = train_test_split(patients, test_size=0.2, random_state=42)
train_df = data[data.patient_id.isin(tr_p)].reset_index(drop=True)
test_df  = data[data.patient_id.isin(te_p)].reset_index(drop=True)
assert len(train_df) == 1837 and len(test_df) == 461, (len(train_df), len(test_df))

groups_tr = train_df.patient_id.to_numpy(dtype=str)         # numpy array for GroupKFold / GroupShuffleSplit

fold = np.full(len(train_df), -1)
for k, (_, va) in enumerate(GroupKFold(n_splits=5).split(train_df, train_df[TARGET], groups=groups_tr)):
    fold[va] = k
train_df['fold'] = fold

le = LabelEncoder().fit(train_df[TARGET])
y_tr = le.transform(train_df[TARGET]); y_te = le.transform(test_df[TARGET])
C = len(le.classes_)

json.dump({
    'train_patients': sorted(map(str, tr_p)), 'test_patients': sorted(map(str, te_p)),
    'train_img_ids': train_df.img_id.tolist(), 'test_img_ids': test_df.img_id.tolist(),
    'fold_by_img_id': dict(zip(train_df.img_id, fold.tolist())),
    'classes': list(le.classes_),
}, open('split_v1.json', 'w'), indent=1)
print('train/test:', len(train_df), len(test_df), '| fold sizes:', np.bincount(fold), '| classes:', list(le.classes_))

train/test: 1837 461 | fold sizes: [368 368 367 367 367] | classes: ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK']


## 1b. Tabular branch — XGBoost (softprob), OOF on the same folds + final model

In [11]:
X_tr = pd.get_dummies(train_df[FEATURES], dummy_na=True).astype(float)
X_te = pd.get_dummies(test_df[FEATURES], dummy_na=True).reindex(columns=X_tr.columns, fill_value=0).astype(float)
print('tabular design matrix:', X_tr.shape, X_te.shape)

def make_xgb():
    return XGBClassifier(objective='multi:softprob', num_class=C, eval_metric='mlogloss', random_state=42)

oof_xgb = np.zeros((len(train_df), C))
for k in range(5):
    tr, va = fold != k, fold == k
    oof_xgb[va] = make_xgb().fit(X_tr[tr], y_tr[tr]).predict_proba(X_tr[va])

xgb_final = make_xgb().fit(X_tr, y_tr)
test_xgb = xgb_final.predict_proba(X_te)
joblib.dump({'model': xgb_final, 'columns': list(X_tr.columns)}, 'xgb_final_v1.joblib')
print('XGB OOF macro-F1: %.3f | test macro-F1: %.3f' % (
    f1_score(y_tr, oof_xgb.argmax(1), average='macro'), f1_score(y_te, test_xgb.argmax(1), average='macro')))

tabular design matrix: (1837, 40) (461, 40)
XGB OOF macro-F1: 0.550 | test macro-F1: 0.523


## 1c. Image cache (raw uint8, 224×224) — augmentation is applied on the fly during training

In [12]:
img_index = build_image_index(IMG_ROOT)
assert train_df.img_id.isin(img_index).all() and test_df.img_id.isin(img_index).all()

def load_uint8(df):
    arr = np.zeros((len(df), 3, 224, 224), dtype=np.uint8)
    for i, r in enumerate(tqdm(df.itertuples(), total=len(df), desc='loading')):
        arr[i] = np.asarray(Image.open(img_index[r.img_id]).convert('RGB').resize((224, 224))).transpose(2, 0, 1)
    return torch.from_numpy(arr)

IMG_TR, IMG_TE = load_uint8(train_df), load_uint8(test_df)
torch.save({'tr': IMG_TR, 'te': IMG_TE}, 'images_uint8_224.pt')
print(IMG_TR.shape, IMG_TE.shape)

loading: 100%|██████████| 461/461 [00:18<00:00, 25.15it/s]


RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 576 vs 470

## 1d. ViT helpers

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
AUG = v2.Compose([
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(),
    v2.RandomAffine(degrees=25, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
])

def prep(x_uint8, train):
    x = x_uint8.to(device, non_blocking=True).float() / 255.0
    if train:
        x = torch.stack([AUG(xi) for xi in x])        # independent random params per image
    return (x - MEAN) / STD

def make_vit():
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False                        # stage 1: frozen backbone
    m.heads.head = nn.Linear(m.heads.head.in_features, C)
    return m.to(device)

@torch.no_grad()
def predict_proba(m, X_uint8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_uint8), bs):
        out.append(torch.softmax(m(prep(X_uint8[i:i+bs], False)), dim=1).cpu())
    return torch.cat(out).numpy()

## 1e. `train_vit` — Imrie et al. two-stage recipe (used for every fold AND the final model)

In [ ]:
def train_vit(X_uint8, y, groups, seed=42, bs=32, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    # inner 10% patient-level validation split, used only for early stopping
    tr_i, va_i = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=seed).split(X_uint8, y, groups))
    tr_i, va_i = torch.tensor(tr_i), torch.tensor(va_i)
    y_t = torch.tensor(y, dtype=torch.long)

    m = make_vit()
    cw = torch.tensor(compute_class_weight('balanced', classes=np.arange(C), y=y[tr_i.numpy()]),
                      dtype=torch.float, device=device)
    crit = nn.CrossEntropyLoss(weight=cw)

    @torch.no_grad()
    def val_loss():
        m.eval(); tot = 0.0
        for i in range(0, len(va_i), 64):
            idx = va_i[i:i+64]
            tot += crit(m(prep(X_uint8[idx], False)), y_t[idx].to(device)).item() * len(idx)
        return tot / len(va_i)

    def run(params, lr, max_epochs, patience, tag):
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
        best, best_state, bad = np.inf, copy.deepcopy(m.state_dict()), 0
        for ep in range(max_epochs):
            m.train()
            perm = tr_i[torch.randperm(len(tr_i))]
            for i in range(0, len(perm), bs):
                idx = perm[i:i+bs]
                opt.zero_grad(set_to_none=True)
                loss = crit(m(prep(X_uint8[idx], True)), y_t[idx].to(device))
                loss.backward(); opt.step()
            vl = val_loss()
            improved = vl < best
            if verbose and (ep % 5 == 0 or improved):
                print(f'    [{tag}] ep {ep:3d}  inner-val loss {vl:.4f}' + ('  *' if improved else ''))
            if improved:
                best, best_state, bad = vl, copy.deepcopy(m.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    break
        m.load_state_dict(best_state)
        return best

    b1 = run(m.heads.parameters(), lr=1e-4, max_epochs=50, patience=20, tag='stage1 head')
    for p in m.parameters():
        p.requires_grad = True                          # stage 2: unfreeze everything
    b2 = run(m.parameters(), lr=1e-6, max_epochs=100, patience=20, tag='stage2 full')
    f1 = f1_score(y[va_i.numpy()], predict_proba(m, X_uint8[va_i]).argmax(1), average='macro')
    if verbose:
        print(f'  best inner-val loss: stage1 {b1:.4f} -> stage2 {b2:.4f} | inner-val macro-F1 {f1:.3f}')
    return m

## 1f. ViT OOF — 5 folds, same recipe each time

In [ ]:
oof_vit = np.zeros((len(train_df), C))
for k in range(5):
    print(f'== fold {k}')
    tr = np.where(fold != k)[0]; va = np.where(fold == k)[0]
    m = train_vit(IMG_TR[tr], y_tr[tr], groups_tr[tr])
    oof_vit[va] = predict_proba(m, IMG_TR[va])
    print(f'  fold {k} OOF macro-F1: {f1_score(y_tr[va], oof_vit[va].argmax(1), average="macro"):.3f}')
    del m; torch.cuda.empty_cache()
print('ViT OOF macro-F1 (all folds): %.3f' % f1_score(y_tr, oof_vit.argmax(1), average='macro'))
np.save('oof_vit_tmp.npy', oof_vit)   # checkpoint in case the next cell crashes

## 1g. Final ViT on all training rows — this is the model we explain later

In [ ]:
vit_final = train_vit(IMG_TR, y_tr, groups_tr)
torch.save(vit_final.state_dict(), 'vit_final_v1.pt')
test_vit = predict_proba(vit_final, IMG_TE)
print('ViT test macro-F1: %.3f' % f1_score(y_te, test_vit.argmax(1), average='macro'))

## 1h. Save everything and print the four reports

In [ ]:
np.savez('base_outputs_v1.npz',
         oof_xgb=oof_xgb, oof_vit=oof_vit, test_xgb=test_xgb, test_vit=test_vit,
         y_train=y_tr, y_test=y_te, classes=np.array(le.classes_), fold=fold)

for name, p, y in [('XGB OOF', oof_xgb, y_tr), ('ViT OOF', oof_vit, y_tr),
                   ('XGB test', test_xgb, y_te), ('ViT test', test_vit, y_te)]:
    print('=' * 20, name)
    print(classification_report(y, p.argmax(1), target_names=le.classes_, zero_division=0))